[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/10_deep_learning/_assets/03_finetuning_lora.ipynb)

# Notebook 3 · Fine-tuning de imágenes con LoRA: tu dataset

Vas a adaptar **Stable Diffusion 1.5** a tus imágenes con **LoRA**. Compararás el modelo base con el adaptado, verás qué cambió en sus capas y descargarás tu adaptador.

1. En Colab, abre «Entorno de ejecución → Cambiar tipo de entorno», elige **T4 GPU** y guarda.
2. Arrastra tu `imagenes.zip` al panel **📁 Archivos** de la izquierda y espera a que termine la carga.
3. Selecciona «Entorno de ejecución → **Ejecutar todo**». Puedes seguir el mismo recorrido sin subir un zip, con el demo de caras de anime.

Elige `MODO = "corto"` para clase. En una T4, reserva unos 20–25 minutos para la ejecución completa, incluidas las descargas. El tiempo es orientativo. El modo `"largo"` permite usar más datos y dedica más pasos al entrenamiento y más muestras a la evaluación.

| Sección | Qué haces |
|---|---|
| 01 · Configuración | Eliges el modo y compruebas la GPU. |
| 02 · Tu zip | Revisas y preparas los datos. |
| 03 · Antes | Generas imágenes con el modelo base. |
| 04 · LoRA por dentro | Descompones la actualización y cuentas sus parámetros. |
| 05 · Entrenamiento | Ajustas el adaptador y lees las curvas de loss. |
| 06 · Inspección | Ves el rango de la actualización, las capas y el efecto de cambiar su escala. |
| 07 · Métricas | Comparas base y LoRA en **validation**. |
| 08 · Testeo | Evalúas en **test una sola vez**, al final, y guardas el adaptador. |

## 01 · Instalar y configurar

Elige `MODO` en la primera línea de código. `LARGO` indica si activaste el modo largo. La celda instala los paquetes que faltan y conserva las versiones ya instaladas.

| Paquete | Para qué lo usamos |
|---|---|
| `diffusers` | Cargar Stable Diffusion y sus schedulers de difusión. |
| `transformers` | Procesar captions y obtener embeddings de CLIP. |
| `peft` | Insertar el adaptador LoRA en el UNet. |
| `accelerate` | Apoyar la carga de los pesos. |

In [ ]:
MODO = "corto"   # "corto" (≈ 20–25 min en una T4) o "largo" (≈ 50–60 min)
LARGO = MODO == "largo"

# Instala solo lo que falta; no actualiza lo que Colab ya trae.
import importlib.util
import subprocess
import sys

FALTAN = [p for p in ["diffusers", "peft", "accelerate", "transformers"]
          if importlib.util.find_spec(p) is None]
if FALTAN:
    print("Instalando:", ", ".join(FALTAN))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *FALTAN],
                   check=True)
else:
    print("No falta nada que instalar.")
print("MODO =", MODO, "· LARGO =", LARGO)

- `DEVICE` indica dónde se ejecuta el modelo. Si Colab avisa que estás en CPU, activa la **T4 GPU** y vuelve a ejecutar todo.
- `DTYPE` indica la precisión de los pesos base. En GPU se cargan en fp16, que ocupa la mitad por parámetro que fp32. En CPU se usa fp32.
- `RAPIDO=1` es una prueba de ejecución en CPU con modelos diminutos. Sus imágenes y métricas sirven para revisar el flujo.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import diffusers
import peft
import transformers

# RAPIDO=1: prueba local en CPU con modelos diminutos (sus imágenes son ruido).
RAPIDO = os.environ.get("RAPIDO") == "1"
HAY_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAY_GPU and not RAPIDO else "cpu")
CUDA = DEVICE.type == "cuda"
DTYPE = torch.float16 if CUDA else torch.float32   # la T4 no tiene bf16

if not HAY_GPU and not RAPIDO:
    print("=" * 72)
    print("  ¡AVISO! ESTE ENTORNO NO TIENE GPU.")
    print("  Stable Diffusion en CPU tardaría horas. En Colab:")
    print("    Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")
    print("    → Guardar, y luego Entorno de ejecución → Ejecutar todo.")
    print("=" * 72)

for nombre, mod in [("torch", torch), ("diffusers", diffusers),
                    ("peft", peft), ("transformers", transformers)]:
    print(f"{nombre:14s}{mod.__version__}")
print("DEVICE =", DEVICE, "· DTYPE =", DTYPE, "· RAPIDO =", RAPIDO)
if HAY_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} · {props.total_memory / 2**30:.1f} GiB")

La celda muestra `P`, la configuración efectiva de tu corrida.

| Claves de `P` | Qué controlan |
|---|---|
| `res`, `max_imagenes` | Resolución y tope de imágenes tomadas del zip. |
| `rango`, `lr` | Rango del adaptador y learning rate. |
| `batch`, `pasos` | Imágenes por batch y actualizaciones programadas. |
| `val_cada`, `n_val_ruido` | Frecuencia de evaluación y cantidad de pares de ruido fijos. |
| `semillas`, `pasos_inferencia` | Semillas de las mallas y pasos para generar cada imagen. |
| `n_gen` | Imágenes generadas por modelo para evaluar los captions de cada split. |

`semilla(s)` fija el azar de Python, NumPy y PyTorch. Al generar, cada pareja de prompt y semilla fija el punto de partida de la comparación entre base y LoRA.

In [ ]:
# El modelo que ajustamos y el CLIP que mide las imágenes.
if RAPIDO:
    MODELO = "hf-internal-testing/tiny-stable-diffusion-pipe"
    MODELO_CLIP = "hf-internal-testing/tiny-random-CLIPModel"
else:
    MODELO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    MODELO_CLIP = "openai/clip-vit-base-patch32"

# Tamaños por modo. pasos = actualizaciones del adaptador.
if RAPIDO:
    P = dict(res=32, max_imagenes=60, demo_personajes=0, demo_por_personaje=0,
             rango=8, lr=1e-4, batch=2, pasos=4, val_cada=2, n_val_ruido=2,
             semillas=[0, 1], n_gen=3, pasos_inferencia=2)
elif LARGO:
    P = dict(res=512, max_imagenes=3000, demo_personajes=40,
             demo_por_personaje=15, rango=8, lr=1e-4, batch=4, pasos=1200,
             val_cada=100, n_val_ruido=16, semillas=[0, 1, 2, 3], n_gen=16,
             pasos_inferencia=30)
else:
    P = dict(res=512, max_imagenes=1000, demo_personajes=20,
             demo_por_personaje=10, rango=8, lr=1e-4, batch=4, pasos=400,
             val_cada=50, n_val_ruido=8, semillas=[0, 1], n_gen=8,
             pasos_inferencia=20)
display(pd.Series(P, name="valor").to_frame())

# Un color fijo por conjunto, como en los notebooks 1 y 2.
COLORES = {"train": "#1f77b4", "validation": "#ff7f0e", "test": "#2ca02c",
           "base": "#7f7f7f", "LoRA": "#9467bd"}


def semilla(s: int):
    """Fija las semillas de random, NumPy y PyTorch (CPU y GPU)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)        # también siembra la GPU


semilla(0)
plt.style.use("default")
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 80, "savefig.dpi": 80,
                     "axes.spines.top": False, "axes.spines.right": False})
try:                            # en Colab: figuras en JPEG, el notebook pesa menos
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 80})
except Exception:
    pass

## 02 · Tu zip

| Dónde está tu zip | Qué haces |
|---|---|
| En tu computadora | Arrástralo al panel **📁 Archivos** de la izquierda y espera a que termine la carga. |
| En Google Drive | Descomenta `drive.mount(...)` y pega la ruta del zip en `RUTA_ZIP`. |
| Si falla el arrastre | Descomenta `files.upload()` para abrir el selector de archivos. |

Con `RUTA_ZIP = ""`, se busca `imagenes*.zip` en `/content` y en la carpeta de trabajo. Si hay varios, se elige el más reciente. Si el tuyo tiene otro nombre, escribe su ruta en `RUTA_ZIP`. Si solo hay coincidencias con `texto*.zip`, la celda te dirige al notebook 4.

In [ ]:
RUTA_ZIP = ""   # vacío: lo busco yo. O escribe la ruta, p. ej.
#                 "/content/drive/MyDrive/imagenes.zip"

# Alternativas (quita el # de la que quieras usar):
# from google.colab import files; files.upload()                 # un botón
# from google.colab import drive; drive.mount("/content/drive")  # tu Drive

from pathlib import Path

NB4 = ("https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/"
       "main/course/10_deep_learning/_assets/04_finetuning_llm.ipynb")


def buscar(patron):
    """Zips que coinciden con `patron` en /content y en la carpeta actual,
    del más reciente al más viejo."""
    vistos = {p.resolve() for c in (Path("/content"), Path.cwd())
              if c.is_dir() for p in c.glob(patron) if p.is_file()}
    return sorted(vistos, key=lambda p: p.stat().st_mtime, reverse=True)


if RUTA_ZIP:
    if not Path(RUTA_ZIP).is_file():
        raise FileNotFoundError(f"No existe {RUTA_ZIP!r}. Revisa la ruta en el "
                                "panel 📁 (clic derecho → Copiar ruta).")
elif buscar("imagenes*.zip"):          # «imagenes (1).zip» también cuenta
    RUTA_ZIP = str(buscar("imagenes*.zip")[0])
elif buscar("texto*.zip"):
    raise ValueError(
        f"Encontré {buscar('texto*.zip')[0].name}: es un dataset de TEXTO. "
        f"Este notebook es el de imágenes. Abre el notebook 4:\n{NB4}")
ES_DEMO = not RUTA_ZIP
if ES_DEMO:
    otros = [p.name for p in buscar("*.zip")]
    if otros:
        print("Hay estos zip, pero ninguno se llama imagenes*.zip:", otros)
        print("Si uno es el tuyo, escribe su ruta en RUTA_ZIP y corre de nuevo.")
    print("No encontré tu imagenes.zip: uso el dataset DEMO (caras de anime).")
else:
    print("Uso tu zip:", RUTA_ZIP)

Si dejas la ruta vacía y la búsqueda no encuentra ninguno de los dos tipos de zip, la celda crea `demo/imagenes.zip` con el formato de la tarea. En una corrida normal, toma caras de anime de una versión fija de `ComputerVisionAnimeProject/AnimeFaceColorization`, con muestreo reproducible. En `RAPIDO=1`, dibuja óvalos de colores.

El demo usa el caption `an anime face` para todas las imágenes. Sus prompts piden variaciones de pelo, lentes o edad para explorar cómo responde el modelo. Si aparece un aviso de resolución, las imágenes se ampliaron durante la preparación.

In [ ]:
import csv
import re
import shutil
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageDraw

DATASET_DEMO = "ComputerVisionAnimeProject/AnimeFaceColorization"
REV_DEMO = "e8deff76ecee132a5eead1b03941789ba60098fa"   # versión fija
PROMPTS_DEMO = ["an anime face with short green hair",
                "an anime face wearing glasses",
                "an anime face, smiling, red eyes",
                "an anime face of an old man"]


def caras_reales():
    """N caras por personaje, personajes y caras elegidos con semilla 0."""
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()

    def listar(carpeta):
        return [x.path for x in api.list_repo_tree(
            DATASET_DEMO, repo_type="dataset", path_in_repo=carpeta,
            revision=REV_DEMO)]

    def bajar(ruta):
        return Image.open(hf_hub_download(
            DATASET_DEMO, ruta, repo_type="dataset", revision=REV_DEMO))

    # Solo las carpetas reference/NNN_<personaje>: son las caras.
    personajes = sorted(r for r in listar("reference")
                        if re.match(r"^\d{3}_", r.split("/")[-1]))
    azar = random.Random(0)
    elegidos = sorted(azar.sample(personajes, P["demo_personajes"]))
    with ThreadPoolExecutor(16) as pool:
        rutas = []
        for lista in pool.map(listar, elegidos):
            pngs = sorted(r for r in lista if r.endswith(".png"))
            rutas += azar.sample(pngs, min(len(pngs), P["demo_por_personaje"]))
        return [im.convert("RGB") for im in pool.map(bajar, rutas)]


def caras_de_juguete(n=50):
    """Para la prueba en CPU: óvalos de colores, sin descargar nada."""
    azar, salida = random.Random(0), []
    for _ in range(n):
        im = Image.new("RGB", (64, 64), tuple(azar.choices(range(256), k=3)))
        ImageDraw.Draw(im).ellipse((12, 8, 52, 58),
                                   fill=tuple(azar.choices(range(256), k=3)))
        salida.append(im)
    return salida

In [ ]:
def armar_demo(carpeta=Path("demo/imagenes")):
    """Escribe el demo en el formato de la tarea y lo empaqueta en un zip."""
    shutil.rmtree(carpeta, ignore_errors=True)
    (carpeta / "images").mkdir(parents=True)
    caras = caras_de_juguete() if RAPIDO else caras_reales()
    with open(carpeta / "metadata.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["file_name", "text"])
        for i, im in enumerate(caras):
            im.save(carpeta / f"images/{i:04d}.jpg", quality=95)
            w.writerow([f"images/{i:04d}.jpg", "an anime face"])
    (carpeta / "prompts.txt").write_text("\n".join(PROMPTS_DEMO) + "\n",
                                         encoding="utf-8")
    (carpeta / "LEEME.md").write_text(
        "# Dataset demo\n"
        "- **Qué quiero que aprenda el modelo:** el estilo «cara de anime»\n"
        "- **Ruta:** imágenes\n"
        f"- **Fuente:** https://huggingface.co/datasets/{DATASET_DEMO}\n"
        "- **Licencia:** CC-BY-4.0 (las caras son de personajes con derechos"
        " de sus autores; solo para estudiar el método)\n"
        f"- **Cuántos ejemplos y cómo los elegí:** {len(caras)} caras, "
        "personajes y caras al azar con semilla 0\n", encoding="utf-8")
    return shutil.make_archive(str(carpeta), "zip", carpeta)


if ES_DEMO:
    RUTA_ZIP = armar_demo()
    print("Demo armado en", RUTA_ZIP)
else:
    print("(tienes tu zip: no hace falta el demo)")

`revisar_zip` es la función de la tarea. Comprueba los archivos obligatorios, la cantidad de ejemplos y prompts, las columnas del CSV, los captions y que los archivos de imagen se puedan abrir. Si detecta `datos.jsonl` sin `metadata.csv`, el notebook indica que corresponde a la ruta de texto y muestra el enlace al notebook 4.

Si la revisión falla, usa el mensaje para localizar el problema, corrige el zip y vuelve a ejecutar la sección 02. Empaqueta los archivos de modo que `metadata.csv`, `prompts.txt`, `LEEME.md` y `images/` queden en la raíz.

In [ ]:
# Copia literal de la página de la tarea (sección 5).
import csv, io, json, zipfile
from PIL import Image

def revisar_zip(ruta):
    """Dice si el zip sigue el formato de la tarea. No cambia nada."""
    z = zipfile.ZipFile(ruta)
    nombres = set(z.namelist())
    faltan = {"prompts.txt", "LEEME.md"} - nombres
    assert not faltan, f"faltan en la raíz del zip: {sorted(faltan)}"
    prompts = z.read("prompts.txt").decode("utf-8").split("\n")
    prompts = [p for p in prompts if p.strip()]
    assert 3 <= len(prompts) <= 5, f"prompts.txt: {len(prompts)} prompts, van de 3 a 5"
    if "metadata.csv" in nombres:                     # imágenes
        filas = list(csv.DictReader(io.StringIO(z.read("metadata.csv").decode("utf-8"))))
        assert filas and set(filas[0]) == {"file_name", "text"}, "columnas: file_name,text"
        assert len(filas) >= 50, f"solo {len(filas)} imágenes; mínimo 50"
        for fila in filas:
            assert fila["file_name"] in nombres, f"no está {fila['file_name']}"
            assert fila["text"].strip(), f"{fila['file_name']} sin texto"
            Image.open(z.open(fila["file_name"])).verify()
        print(f"OK · imágenes: {len(filas)} · prompts: {len(prompts)}")
    elif "datos.jsonl" in nombres:                    # texto
        lineas = z.read("datos.jsonl").decode("utf-8").splitlines()
        ejemplos = [json.loads(l) for l in lineas if l.strip()]
        assert len(ejemplos) >= 200, f"solo {len(ejemplos)} ejemplos; mínimo 200"
        for i, ej in enumerate(ejemplos, 1):
            assert set(ej) == {"prompt", "completion"}, f"línea {i}: llaves {sorted(ej)}"
            assert ej["completion"].strip(), f"línea {i}: completion vacío"
        print(f"OK · ejemplos: {len(ejemplos)} · prompts: {len(prompts)}")
    else:
        raise AssertionError("falta metadata.csv (imágenes) o datos.jsonl (texto)")

In [ ]:
try:
    with zipfile.ZipFile(RUTA_ZIP) as z:
        NOMBRES_ZIP = z.namelist()
except zipfile.BadZipFile:
    raise ValueError(f"{RUTA_ZIP} no es un zip válido: vuelve a empaquetarlo "
                     "(página de la tarea, sección 5).") from None
if "metadata.csv" not in NOMBRES_ZIP and "datos.jsonl" in NOMBRES_ZIP:
    raise ValueError("Este zip es un dataset de TEXTO (trae datos.jsonl). "
                     f"Este notebook es el de imágenes. Abre el notebook 4:\n{NB4}")
try:
    revisar_zip(RUTA_ZIP)
except AssertionError as e:
    pista = ""
    if any(n.endswith("/metadata.csv") for n in NOMBRES_ZIP):
        pista = ("\nPista: metadata.csv está dentro de una carpeta. Comprime "
                 "los archivos, no la carpeta.")
    raise ValueError(f"Tu zip no sigue el formato de la tarea: {e}{pista}\n"
                     "Corrígelo con la página de la tarea (secciones 2 y 5) "
                     "y vuelve a subirlo.") from None

| Paso | Qué hace |
|---|---|
| Descomprimir | Extrae el zip en `datos_zip/` y lee el CSV, los prompts y la ficha. |
| Muestrear | Si el zip supera `P["max_imagenes"]`, toma una muestra con semilla fija. |
| Preparar | Corrige la orientación EXIF, convierte a RGB y hace un recorte cuadrado al centro del tamaño `P["res"]`. |
| Deduplicar | Descarta imágenes con la misma huella al reducirlas. Así evita repartir copias detectadas entre los splits. |
| Partir | Baraja con semilla fija y reparte aproximadamente 80 % para **train**, 10 % para **validation** y 10 % para **test**. |

Cada imagen conserva el caption de la columna `text`. **Train** actualiza el adaptador, **validation** orienta tus decisiones y **test** queda reservado para la evaluación final.

In [ ]:
import hashlib

from PIL import ImageOps

CARPETA_ZIP = Path("datos_zip")             # aquí se descomprime tu zip
shutil.rmtree(CARPETA_ZIP, ignore_errors=True)
with zipfile.ZipFile(RUTA_ZIP) as z:
    z.extractall(CARPETA_ZIP)
with open(CARPETA_ZIP / "metadata.csv", encoding="utf-8") as f:
    FILAS = list(csv.DictReader(f))
PROMPTS = [p.strip() for p in (CARPETA_ZIP / "prompts.txt")
           .read_text(encoding="utf-8").split("\n") if p.strip()]
LEEME = (CARPETA_ZIP / "LEEME.md").read_text(encoding="utf-8")
if len(FILAS) > P["max_imagenes"]:
    print(f"Tu zip trae {len(FILAS)} imágenes; tomo {P['max_imagenes']} "
          "al azar con semilla 0 (P['max_imagenes']).")
    FILAS = random.Random(0).sample(FILAS, P["max_imagenes"])


def abrir(fila):
    """Abre, endereza (EXIF), recorta al centro y deja a P["res"] px."""
    im = Image.open(CARPETA_ZIP / fila["file_name"])
    im.draft("RGB", (P["res"], P["res"]))   # un JPEG enorme se lee ya reducido
    im = ImageOps.exif_transpose(im).convert("RGB")
    lado = min(im.size)
    return ImageOps.fit(im, (P["res"], P["res"]), Image.BICUBIC), lado


def huella(im):
    """Hash de la imagen reducida a 16×16: detecta duplicados casi exactos."""
    return hashlib.md5(im.resize((16, 16), Image.BILINEAR).tobytes()).hexdigest()


with ThreadPoolExecutor(8) as pool:
    ABIERTAS = list(pool.map(abrir, FILAS))
# Deduplicar ANTES de partir: una imagen repetida en train y en test
# haría que test premie memorizar.
vistas, UNICAS = set(), []
for fila, (im, lado) in zip(FILAS, ABIERTAS):
    if huella(im) not in vistas:
        vistas.add(huella(im))
        UNICAS.append((im, fila["text"].strip(), lado))
chicas = sum(lado < 512 for _, _, lado in UNICAS)

# Partición 80 / 10 / 10 con semilla 0.
orden = list(range(len(UNICAS)))
random.Random(0).shuffle(orden)
n_val = n_test = max(1, round(0.1 * len(orden)))
PARTES = {"train": orden[n_val + n_test:], "validation": orden[:n_val],
          "test": orden[n_val:n_val + n_test]}
IMGS = {k: [UNICAS[i][0] for i in v] for k, v in PARTES.items()}
TEXTOS = {k: [UNICAS[i][1] for i in v] for k, v in PARTES.items()}
display(pd.DataFrame({k: {"imágenes": len(v),
                          "captions distintos": len(set(TEXTOS[k]))}
                      for k, v in IMGS.items()}))
print(f"duplicadas descartadas: {len(FILAS) - len(UNICAS)}")
if chicas and not RAPIDO:
    print(f"Ojo: {chicas} imágenes tienen el lado corto < 512 px; se agrandan "
          "y quedan más suaves. El modelo también aprende esa suavidad.")

**Cómo leer la malla.** Cada imagen de train lleva su caption. Revisa que el recorte conserve el motivo, y que el caption describa lo que quedó en la imagen. Debajo aparecen los prompts de prueba y la ficha de tu dataset.

In [ ]:
import math
import textwrap


def malla(imagenes, filas, titulo, etiquetas=None):
    """Malla de imágenes (≤ 256 px). etiquetas: una por imagen (título
    chico) o una por fila (a la izquierda)."""
    n = len(imagenes)
    cols = max(1, math.ceil(n / filas))
    fig, ejes = plt.subplots(filas, cols, figsize=(1.9 * cols, 2.0 * filas),
                             squeeze=False, dpi=80)
    for i, eje in enumerate(ejes.flat):
        eje.axis("off")
        if i >= n:
            continue
        im = imagenes[i].copy()
        im.thumbnail((256, 256))
        eje.imshow(im)
        if etiquetas is not None and len(etiquetas) == n:
            eje.set_title(textwrap.shorten(etiquetas[i], 28), fontsize=7)
    if etiquetas is not None and len(etiquetas) == filas and filas != n:
        for f in range(filas):
            ejes[f, 0].text(-0.05, 0.5, textwrap.fill(etiquetas[f], 22),
                            fontsize=8, ha="right", va="center",
                            transform=ejes[f, 0].transAxes)
    fig.suptitle(titulo, fontsize=10)
    plt.tight_layout()
    plt.show()


k = min(8, len(IMGS["train"]))
malla(IMGS["train"][:k], filas=1 if k <= 4 else 2,
      titulo=f"Imágenes de train (de {len(IMGS['train'])}), con su caption",
      etiquetas=TEXTOS["train"][:k])
print("Tus prompts de prueba (prompts.txt):")
for p in PROMPTS:
    print("  ·", p)
print("\nTu ficha (LEEME.md):\n" + LEEME)

## 03 · Antes: el modelo base

Primero genera con Stable Diffusion tal como viene. Estas imágenes serán tu referencia para juzgar el efecto del adaptador.

| Pieza | Qué hace |
|---|---|
| VAE | Codifica una imagen en un latent $x_0$ y convierte un latent generado en píxeles. |
| Text encoder (CLIP) | Convierte el prompt en los embeddings $c$ que condicionan al UNet. |
| UNet | Estima el ruido a partir del latent ruidoso $x_t$, el paso $t$ y los embeddings $c$. |

La celda carga el pipeline e imprime cuántos parámetros tiene cada pieza. Los pesos base quedan congelados y el adaptador se añadirá al **UNet**. El pipeline se carga con el *safety checker* desactivado.

In [ ]:
import gc

from diffusers import (AutoencoderKL, DPMSolverMultistepScheduler,
                       StableDiffusionPipeline, UNet2DConditionModel)
from transformers import CLIPTextModel, CLIPTokenizer


def cargar_pipe():
    """StableDiffusionPipeline armado pieza por pieza (fp16 en la GPU)."""
    piezas = dict(
        vae=AutoencoderKL.from_pretrained(MODELO, subfolder="vae",
                                          torch_dtype=DTYPE),
        text_encoder=CLIPTextModel.from_pretrained(
            MODELO, subfolder="text_encoder", torch_dtype=DTYPE),
        tokenizer=CLIPTokenizer.from_pretrained(MODELO, subfolder="tokenizer"),
        unet=UNet2DConditionModel.from_pretrained(
            MODELO, subfolder="unet", torch_dtype=DTYPE),
        scheduler=DPMSolverMultistepScheduler.from_pretrained(
            MODELO, subfolder="scheduler"))
    pipe = StableDiffusionPipeline(**piezas, safety_checker=None,
                                   feature_extractor=None,
                                   requires_safety_checker=False).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    pipe.vae.enable_slicing()          # decodifica de una en una: menos VRAM
    for pieza in (pipe.vae, pipe.text_encoder, pipe.unet):
        pieza.requires_grad_(False)    # nada del modelo base se entrena
    return pipe


PIPE = cargar_pipe()
UNET = PIPE.unet                       # el que recibirá el adaptador LoRA
for nombre in ("unet", "vae", "text_encoder"):
    n = sum(p.numel() for p in getattr(PIPE, nombre).parameters())
    print(f"{nombre:13s}{n / 1e6:8.1f} M parámetros")

`combinar(prompts, semillas)` empareja cada prompt con cada semilla. `generar` produce una imagen por pareja. Usa `lora=False` para el modelo base y, después del entrenamiento, `lora=True` para activar el adaptador. El argumento `escala` controla su intensidad.

Al comparar dos salidas con el mismo prompt y la misma semilla, ambas parten del mismo ruido inicial.

In [ ]:
def tiene_lora():
    """¿Ya tiene el UNet un adaptador? (antes de la sección 05, no)."""
    return bool(getattr(UNET, "peft_config", None))


def usar_lora(si):
    """Prende (si=True) o apaga el adaptador. W0 nunca se toca."""
    if tiene_lora():
        (UNET.enable_adapters if si else UNET.disable_adapters)()
    elif si:
        raise RuntimeError("Aún no hay adaptador: corre la sección 05.")


def combinar(prompts, semillas):
    """Todas las parejas (prompt, semilla), prompt por prompt."""
    return [(p, s) for p in prompts for s in semillas]


@torch.no_grad()
def generar(pares, lora=False, escala=1.0):
    """Una imagen por pareja (prompt, semilla). lora=False → el modelo base.
    escala = λ multiplica a ΔW (solo con lora=True)."""
    usar_lora(lora)
    UNET.eval()
    # sin adaptador no se pasa scale: diffusers lo exige solo con LoRA
    extra = {"cross_attention_kwargs": {"scale": escala}} if lora else {}
    salida = []
    for i in range(0, len(pares), 2):  # de 2 en 2: cabe en la T4
        lote = pares[i:i + 2]
        gens = [torch.Generator("cpu").manual_seed(s) for _, s in lote]
        salida += PIPE([p for p, _ in lote], generator=gens,
                       num_inference_steps=P["pasos_inferencia"],
                       height=P["res"], width=P["res"], **extra).images
    return salida

**Cómo leer la malla.** Cada fila corresponde a un prompt de `prompts.txt` y cada columna a una semilla. Mira qué elementos del prompt aparecen y cómo se ven el color, la textura y el encuadre. En la sección 08 volverás a ver estas imágenes junto a las de LoRA.

In [ ]:
ANTES = generar(combinar(PROMPTS, P["semillas"]))
malla(ANTES, filas=len(PROMPTS), titulo="ANTES · el modelo base con tus prompts",
      etiquetas=PROMPTS)                   # una etiqueta por fila

**Cómo leer la malla.** Arriba tienes ejemplos reales de validation. Debajo de cada uno ves lo que genera el modelo base con su caption. Si el caption es amplio, muchas imágenes distintas pueden corresponderle. Compara rasgos de tema y estilo entre las filas.

In [ ]:
# Tres imágenes reales de validation (arriba) y lo que el base genera con
# su mismo caption (abajo). Semillas 1000, 1001, … : ninguna otra celda las usa.
k = min(3, len(IMGS["validation"]))
pares_val = [(TEXTOS["validation"][i], 1000 + i) for i in range(k)]
base_val = generar(pares_val)
malla(IMGS["validation"][:k] + base_val, filas=2,
      titulo="validation: real (arriba) y el base con el mismo caption (abajo)",
      etiquetas=[f"real · {t}" for t, _ in pares_val]
      + [f"base · {t}" for t, _ in pares_val])

## 04 · LoRA por dentro

LoRA (*Low-Rank Adaptation*) congela el peso original $W_0$ y aprende una actualización $\Delta W$ como producto de dos matrices delgadas.

$$
W=W_0+\Delta W,\qquad \Delta W=\frac{\alpha}{r}BA.
$$

| Símbolo | Forma | Papel |
|---|---|---|
| $W_0$ | $d\times k$ | Peso congelado de una capa con $k$ entradas y $d$ salidas. |
| $A$ | $r\times k$ | Proyección entrenable de la entrada a $r$ coordenadas. |
| $B$ | $d\times r$ | Proyección entrenable de esas coordenadas a la salida. |
| $r$, $\alpha$ | Escalares | $r$ limita el rango de la corrección. $\alpha/r$ ajusta su escala y aquí vale uno porque $\alpha=r$. |

- **Inicio.** $A$ se inicializa con valores aleatorios y $B=0$. Por eso al comienzo $\Delta W=0$ y la capa equivale a la original.
- **Forward.** Para una entrada $x$, la parte lineal de la salida es $W_0x+\frac{\alpha}{r}B(Ax)$. La corrección se calcula en dos proyecciones, sin construir la matriz completa $BA$.
- **Rango y parámetros.** Como $Ax$ tiene $r$ coordenadas, $\operatorname{rank}(\Delta W)\le r$. Entrenas $r(d+k)$ parámetros del adaptador en lugar de los $dk$ del peso completo.

La celda cuenta las capas lineales de atención del UNet que recibirán LoRA. `to_q`, `to_k` y `to_v` calculan las proyecciones de *query*, *key* y *value*. `to_out.0` proyecta la salida de la atención.

**Cómo leer la tabla.** Compara la fila de LoRA con la del UNet para ver qué fracción de sus parámetros vas a entrenar. La fila de los pesos $W_0$ seleccionados cuenta solo los pesos de las capas elegidas. Los MB estiman el tamaño de los pesos en fp16 para el modelo base y en fp32 para el adaptador.

In [ ]:
# Las lineales de la atención, como en el notebook 3 y en casi todo LoRA de SD.
MODULOS = ("to_q", "to_k", "to_v", "to_out.0")


def es_objetivo(nombre):
    """La misma regla que peft: el nombre es el módulo o termina en .módulo."""
    return any(nombre == m or nombre.endswith("." + m) for m in MODULOS)


r = P["rango"]
filas = []
for nombre, m in UNET.named_modules():
    nombre = nombre.removesuffix(".base_layer")     # si ya hay adaptador
    if isinstance(m, torch.nn.Linear) and es_objetivo(nombre):
        d, k = m.out_features, m.in_features         # W0 es d × k
        filas.append({"capa": nombre, "d": d, "k": k,
                      "dk": d * k, "r(d+k)": r * (d + k)})
CAPAS = pd.DataFrame(filas).set_index("capa")
N_LORA = int(CAPAS["r(d+k)"].sum())
N_UNET = sum(p.numel() for n, p in UNET.named_parameters() if "lora_" not in n)
N_PIPE = N_UNET + sum(p.numel() for p in PIPE.vae.parameters()) + sum(
    p.numel() for p in PIPE.text_encoder.parameters())

print(f"{len(CAPAS)} capas lineales reciben LoRA. Sus formas (d × k):")
display(CAPAS.groupby(["d", "k"]).size().rename("capas").to_frame())
MB = 1e6
display(pd.DataFrame({
    "parámetros": [N_PIPE, N_UNET, int(CAPAS["dk"].sum()), N_LORA],
    "MB": [2 * N_PIPE / MB, 2 * N_UNET / MB, 2 * CAPAS["dk"].sum() / MB,
           4 * N_LORA / MB],
    "% del UNet": [100 * N_PIPE / N_UNET, 100.0,
                   100 * CAPAS["dk"].sum() / N_UNET, 100 * N_LORA / N_UNET]},
    index=["Stable Diffusion completo (fp16)", "UNet (fp16)",
           "las W0 que se adaptan (fp16)", f"LoRA r={r}: A y B (fp32)"]
).style.format({"parámetros": "{:,.0f}", "MB": "{:,.1f}",
                "% del UNet": "{:.2f} %"}))
print(f"Entrenas {N_LORA:,} números: {100 * N_LORA / N_UNET:.2f} % del UNet.")

## 05 · Entrenar

El UNet aprende a estimar el ruido añadido a una imagen. El scheduler mezcla su latent limpio $x_0$ con ruido gaussiano $\varepsilon\sim\mathcal N(0,I)$ en un paso $t$ elegido al azar.

$$
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon.
$$

$\bar\alpha_t$ fija cuánto queda de la señal original. Conforme aumenta $t$, pesa menos la señal y más el ruido. El UNet recibe $x_t$, $t$ y el embedding $c$ del caption, y predice $\hat\varepsilon=\varepsilon_\theta(x_t,t,c)$. La loss es el error cuadrático medio (MSE) entre $\hat\varepsilon$ y el ruido aplicado, promediado sobre todos los elementos del batch.

La celda precalcula los latents y un embedding por caption distinto porque el VAE y el text encoder están congelados. Guarda los latents en CPU y avisa si algún caption se trunca.

In [ ]:
import time

import torch.nn.functional as F
from diffusers import DDPMScheduler

ESCALA_VAE = PIPE.vae.config.scaling_factor      # 0.18215 en SD 1.5


def a_tensor(imgs):
    """Lista de PIL → tensor (N, 3, H, W) en [-1, 1], como lo espera el VAE."""
    x = torch.stack([torch.from_numpy(np.array(im)) for im in imgs])
    return x.permute(0, 3, 1, 2).float().div(127.5).sub(1)


@torch.no_grad()
def codificar(imgs, lote=8):
    """Latents x_0 = escala · media del VAE, por lotes, guardados en CPU."""
    salida = []
    for i in range(0, len(imgs), lote):
        x = a_tensor(imgs[i:i + lote]).to(DEVICE, DTYPE)
        z = PIPE.vae.encode(x).latent_dist.mode()  # la media: determinista
        salida.append((z * ESCALA_VAE).float().cpu())
    return torch.cat(salida)


LATENTS = {k: codificar(v) for k, v in IMGS.items()}

# Un embedding por caption DISTINTO; CAP[k][i] dice cuál le toca a la imagen i.
CAPTIONS = sorted({t for v in TEXTOS.values() for t in v})
INDICE = {t: i for i, t in enumerate(CAPTIONS)}
CAP = {k: torch.tensor([INDICE[t] for t in v]) for k, v in TEXTOS.items()}
tok = PIPE.tokenizer
with torch.no_grad():
    lotes = []
    for i in range(0, len(CAPTIONS), 32):
        ids = tok(CAPTIONS[i:i + 32], padding="max_length", truncation=True,
                  max_length=tok.model_max_length, return_tensors="pt")
        lotes.append(PIPE.text_encoder(ids.input_ids.to(DEVICE))[0])
    EMBS = torch.cat(lotes).to(DTYPE)            # (captions, 77, D)
largos = sum(len(tok(t).input_ids) > tok.model_max_length for t in CAPTIONS)

for k, z in LATENTS.items():
    print(f"LATENTS[{k!r}]: {tuple(z.shape)}")
print(f"EMBS: {tuple(EMBS.shape)} · {len(CAPTIONS)} captions distintos")
if largos:
    print(f"Ojo: {largos} captions pasan de {tok.model_max_length} tokens; "
          "CLIP solo lee el principio.")

Para comparar la loss en distintos momentos, la celda fija pares $(t,\varepsilon)$. Espacia los $t$ a lo largo del scheduler y genera el ruido con semilla fija. Cada medición vuelve a emparejar las imágenes del split con los mismos pares en el mismo orden. La primera cifra es la loss de validation del modelo base.

In [ ]:
RUIDO = DDPMScheduler.from_pretrained(MODELO, subfolder="scheduler")
T = RUIDO.config.num_train_timesteps
gen = torch.Generator().manual_seed(0)
RUIDO_VAL = (torch.linspace(0, T - 1, P["n_val_ruido"]).round().long(),
             torch.randn((P["n_val_ruido"], *LATENTS["train"].shape[1:]),
                         generator=gen))


@torch.no_grad()
def perdida(conjunto, lora=True):
    """MSE del ruido en `conjunto` con (t, ε) FIJOS: la evaluación i usa la
    imagen i mod N y el par i mod K. Dos mediciones usan la misma vara."""
    usar_lora(lora)
    X, C = LATENTS[conjunto], CAP[conjunto]
    t_fijo, eps_fijo = RUIDO_VAL
    n, k = len(X), len(t_fijo)
    estaba = UNET.training
    UNET.eval()
    total, cuenta = 0.0, 0
    for lote in torch.arange(max(n, k)).split(2 * P["batch"]):
        x0 = X[lote % n].to(DEVICE, DTYPE)
        eps = eps_fijo[lote % k].to(DEVICE, DTYPE)
        t = t_fijo[lote % k].to(DEVICE)
        xt = RUIDO.add_noise(x0, eps, t)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            pred = UNET(xt, t, encoder_hidden_states=EMBS[C[lote % n]]).sample
        total += F.mse_loss(pred.float(), eps.float(), reduction="sum").item()
        cuenta += eps.numel()
    UNET.train(estaba)
    return total / cuenta


print("t fijos de validation:", RUIDO_VAL[0].tolist())
print(f"loss de validation del base: {perdida('validation', lora=False):.4f}")

1. Toma un batch de latents de **train** con sus captions y sortea el ruido y el paso $t$.
2. Forma $x_t$, predice el ruido y calcula la MSE.
3. Calcula los gradientes de $A$ y $B$ y actualiza esas matrices. Los pesos base siguen congelados.

| Técnica | Para qué se usa |
|---|---|
| AdamW | Actualiza los parámetros entrenables con el learning rate de `P["lr"]` y aplica *weight decay*. |
| Warmup y cosine schedule | Sube el learning rate al inicio y luego lo reduce siguiendo una curva coseno. |
| Gradient clipping | Limita la norma de los gradientes antes de actualizar. |
| Mixed precision en GPU | Usa `autocast` para el forward y `GradScaler` para proteger gradientes pequeños. $A$ y $B$ se mantienen en fp32. |
| Gradient checkpointing | Ahorra memoria al recalcular activaciones durante el backward. |

La celda imprime la configuración, estima el tiempo a partir de los primeros pasos y reporta cuántas actualizaciones completó. Si `GradScaler` detecta gradientes no finitos, omite la actualización.

In [ ]:
from diffusers.optimization import get_scheduler
from diffusers.training_utils import cast_training_params
from peft import LoraConfig

if tiene_lora():                      # si vuelves a correr esta celda
    UNET.delete_adapters("default")
semilla(0)
UNET.add_adapter(LoraConfig(r=P["rango"], lora_alpha=P["rango"],
                            init_lora_weights="gaussian",
                            target_modules=list(MODULOS)))
# A y B en fp32 sobre el UNet fp16: GradScaler no desescala gradientes fp16.
cast_training_params(UNET, dtype=torch.float32)
UNET.enable_gradient_checkpointing()  # menos memoria, un poco más de tiempo
ENTRENABLES = [p for p in UNET.parameters() if p.requires_grad]
N_ENTRENABLES = sum(p.numel() for p in ENTRENABLES)
print(f"parámetros entrenables: {N_ENTRENABLES:,} (la cuenta de la sección 04"
      f" dio {N_LORA:,})")
OPT = torch.optim.AdamW(ENTRENABLES, lr=P["lr"], weight_decay=1e-2)
WARMUP = max(1, round(0.05 * P["pasos"]))
SCHED = get_scheduler("cosine", OPT, num_warmup_steps=WARMUP,
                      num_training_steps=P["pasos"])
ESCALADOR = torch.amp.GradScaler("cuda", enabled=CUDA)
print(f"AdamW lr={P['lr']} · warmup {WARMUP} pasos · cosine hasta "
      f"{P['pasos']} · batch {P['batch']}")

In [ ]:
def un_paso(g):
    """Un paso de difusión sobre un batch de train. Devuelve la loss, o
    None si GradScaler encontró inf/NaN y saltó el paso."""
    X, C = LATENTS["train"], CAP["train"]
    i = torch.randint(len(X), (P["batch"],), generator=g)
    x0 = X[i].to(DEVICE, DTYPE)
    eps = torch.randn(x0.shape, generator=g).to(DEVICE, DTYPE)
    t = torch.randint(0, T, (len(i),), generator=g).to(DEVICE)
    xt = RUIDO.add_noise(x0, eps, t)                   # ensuciar: forward
    with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
        pred = UNET(xt, t, encoder_hidden_states=EMBS[C[i]]).sample
    loss = F.mse_loss(pred.float(), eps.float())       # ‖ε − ε̂‖² promedio
    OPT.zero_grad(set_to_none=True)
    ESCALADOR.scale(loss).backward()                   # solo llega a A y B
    ESCALADOR.unscale_(OPT)
    torch.nn.utils.clip_grad_norm_(ENTRENABLES, 1.0)
    escala_previa = ESCALADOR.get_scale()
    ESCALADOR.step(OPT)
    ESCALADOR.update()
    if ESCALADOR.get_scale() < escala_previa:          # inf/NaN: no cuenta
        return None
    SCHED.step()
    return loss.item()


# El punto del paso 0 es el modelo base: B = 0 ⇒ ΔW = 0.
PERDIDAS_TRAIN, PERDIDAS_VAL = [], [(0, perdida("validation"))]
g = torch.Generator().manual_seed(0)
paso, intentos, inicio = 0, 0, time.perf_counter()
usar_lora(True)
UNET.train()
while paso < P["pasos"] and intentos < 2 * P["pasos"]:
    intentos += 1
    loss = un_paso(g)
    if loss is None:
        continue
    paso += 1
    PERDIDAS_TRAIN.append((paso, loss))
    if paso == min(10, P["pasos"]):
        seg = (time.perf_counter() - inicio) / paso
        print(f"{seg:.2f} s/paso → el entrenamiento tardará ≈ "
              f"{seg * P['pasos'] / 60:.1f} min")
    if paso % P["val_cada"] == 0 or paso == P["pasos"]:
        PERDIDAS_VAL.append((paso, perdida("validation")))
        UNET.train()
        print(f"paso {paso:5d} · loss de validation {PERDIDAS_VAL[-1][1]:.4f}")
UNET.eval()
UNET.disable_gradient_checkpointing()
del OPT, SCHED                        # suelta los momentos de AdamW
gc.collect()
if CUDA:
    torch.cuda.empty_cache()
print(f"listo: {paso} pasos en {(time.perf_counter() - inicio) / 60:.1f} min")

**Cómo leer la gráfica.**

- **Train, en azul.** Cada batch trae nuevas imágenes o ruido. Sigue la media móvil para ver la tendencia.
- **Validation, en naranja.** Los pares de ruido son fijos. Lee el cambio frente al punto gris del paso inicial, cuando $B=0$ y el modelo equivale al base.
- Una pérdida de validation menor indica mejor predicción del ruido en ese split. La mejora puede ser pequeña si el modelo base ya predice bien. Si train baja y validation sube de forma sostenida, puede haber sobreajuste. Contrasta la tendencia con las mallas y las métricas de la sección 07.

In [ ]:
pasos_tr, loss_tr = map(np.array, zip(*PERDIDAS_TRAIN))
ventana = max(1, min(25, len(loss_tr) // 5))
media = np.convolve(loss_tr, np.ones(ventana) / ventana, mode="valid")
fig, eje = plt.subplots(figsize=(8, 3.8))
eje.plot(pasos_tr, loss_tr, color=COLORES["train"], lw=0.7, alpha=0.3,
         label="train (cada paso)")
eje.plot(pasos_tr[ventana - 1:], media, color=COLORES["train"], lw=1.8,
         label=f"train (media de {ventana} pasos)")
eje.plot(*zip(*PERDIDAS_VAL), color=COLORES["validation"], lw=2.5,
         marker="o", ms=4, label="validation (t, ε fijos)")
eje.scatter([0], [PERDIDAS_VAL[0][1]], s=90, color=COLORES["base"], zorder=5,
            label="paso 0 = el modelo base")
eje.set(xlabel="paso", ylabel="MSE del ruido",
        title=f"LoRA r={P['rango']}: loss de train y de validation")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()
cambio = PERDIDAS_VAL[-1][1] - PERDIDAS_VAL[0][1]
print(f"validation: {PERDIDAS_VAL[0][1]:.4f} (base) → {PERDIDAS_VAL[-1][1]:.4f}"
      f" (LoRA) · cambio {cambio:+.4f} ({100 * cambio / PERDIDAS_VAL[0][1]:+.1f} %)")

## 06 · Ver LoRA por dentro

Para inspeccionar el adaptador entrenado, la celda sí construye $\Delta W=\frac{\alpha}{r}BA$ para cada capa adaptada. El espectro muestra cuántas direcciones usa la corrección. La norma relativa muestra cuánto pesa frente a $W_0$. La perilla $\lambda$ muestra cómo influye en las imágenes.

In [ ]:
from peft.tuners.lora import LoraLayer


def zona(capa):
    """down / mid / up según el nombre de la capa en el UNet."""
    for z, clave in [("down", "down_blocks"), ("mid", "mid_block"),
                     ("up", "up_blocks")]:
        if clave in capa:
            return z
    return "otra"


def tipo(capa):
    """attn1 = self-attention (la imagen consigo misma), attn2 = cross-
    attention (la imagen mira al texto)."""
    return {"attn1": "self-attention", "attn2": "cross-attention"}.get(
        next((p for p in capa.split(".") if p.startswith("attn")), ""), "otra")


filas = []
for nombre, m in UNET.named_modules():
    if isinstance(m, LoraLayer) and "default" in m.lora_A:
        A = m.lora_A["default"].weight.detach().float().cpu()     # r × k
        B = m.lora_B["default"].weight.detach().float().cpu()     # d × r
        dw = m.scaling["default"] * (B @ A)                        # d × k
        w0 = m.base_layer.weight.detach().float().cpu()
        filas.append({"capa": nombre, "zona": zona(nombre),
                      "tipo": tipo(nombre), "dw": dw,
                      "‖ΔW‖/‖W0‖": (dw.norm() / w0.norm()).item()})
DW = pd.DataFrame(filas).set_index("capa")
print(f"{len(DW)} capas con adaptador · ‖ΔW‖/‖W0‖ mediana "
      f"{DW['‖ΔW‖/‖W0‖'].median():.2e}, máxima {DW['‖ΔW‖/‖W0‖'].max():.2e}")
if DW["‖ΔW‖/‖W0‖"].max() == 0:
    print("ΔW = 0 en todas las capas: el adaptador no se movió (¿corrió la 05?)")

**Valores singulares.** La SVD descompone $\Delta W=\sum_i\sigma_i u_i v_i^\top$, con $\sigma_1\ge\sigma_2\ge\dots\ge0$. El rango exacto es la cantidad de $\sigma_i$ no nulos. Como la actualización pasa por un espacio de dimensión $r$, puede usar hasta $r$ direcciones.

**Cómo leer la gráfica.** Busca valores próximos a cero después de la línea punteada que marca $r$. El eje vertical muestra $\sigma_i/\sigma_1$ en escala logarítmica. La caída puede empezar antes si el adaptador usa menos direcciones. El redondeo puede dejar residuos a la derecha, y una corrección nula deja el cociente sin definir. Se grafica la capa con el mayor cambio relativo y, en modo largo, también la mayor de cada zona disponible.

In [ ]:
def espectro(capa, ax):
    """σ_i / σ_1 de ΔW en escala log; la línea punteada marca r."""
    s = torch.linalg.svdvals(DW.loc[capa, "dw"]).numpy()
    s = s[: 3 * P["rango"]]
    ax.semilogy(np.arange(1, len(s) + 1), np.maximum(s / max(s[0], 1e-30),
                                                       1e-12), "o-", ms=3,
                label=f"{zona(capa)} · {capa.split('.')[-1]} · {tipo(capa)}")
    return s


fig, ax = plt.subplots(figsize=(7.5, 3.6))
mayor = DW["‖ΔW‖/‖W0‖"].idxmax()            # la capa que más se movió
capas = [mayor]
if LARGO:                                   # ➕ una capa más por zona
    capas += [DW[DW["zona"] == z]["‖ΔW‖/‖W0‖"].idxmax()
              for z in ("down", "mid", "up") if (DW["zona"] == z).any()]
for capa in dict.fromkeys(capas):
    s = espectro(capa, ax)
r = P["rango"]
ax.axvline(r, color="gray", ls="--", lw=1)
ax.text(r + 0.3, 1e-10, f"r = {r}", color="gray")
ax.set(xlabel="índice i", ylabel="σ_i / σ_1", ylim=(1e-12, 3),
       title="Valores singulares de ΔW: después de r, nada")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
s = torch.linalg.svdvals(DW.loc[mayor, "dw"])
print(f"{mayor}: ΔW es {tuple(DW.loc[mayor, 'dw'].shape)}; rango numérico "
      f"{torch.linalg.matrix_rank(DW.loc[mayor, 'dw']).item()} (r = {r}); "
      f"σ_{r + 1}/σ_1 = {(s[r] / s[0]).item():.1e}")

La razón $\lVert\Delta W\rVert_F/\lVert W_0\rVert_F$ compara el tamaño de la actualización con el del peso original. La norma de Frobenius $\lVert M\rVert_F=\sqrt{\sum_{i,j} M_{ij}^{\,2}}$ reúne todas las entradas de la matriz en una sola magnitud.

- **Zona.** `down` baja la resolución, `mid` es el centro del UNet y `up` vuelve a subirla.
- **Tipo.** *Self-attention* relaciona posiciones del latent entre sí. *Cross-attention* las relaciona con los tokens del texto.

**Cómo leer la gráfica.** A la izquierda hay una barra por capa adaptada, agrupada por zona. A la derecha están los promedios por zona y tipo de atención. La tabla muestra las capas con mayor cambio relativo. Una barra alta indica una corrección grande frente al peso base de esa capa. El efecto visual también depende de cómo se propague esa corrección por el UNet.

In [ ]:
COLOR_ZONA = {"down": "#9467bd", "mid": "#8c564b", "up": "#17becf",
              "otra": "gray"}
orden = {"down": 0, "mid": 1, "up": 2, "otra": 3}
tabla_dw = DW.sort_values("zona", key=lambda z: z.map(orden), kind="stable")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8),
                             gridspec_kw={"width_ratios": [2, 1]})
a1.bar(range(len(tabla_dw)), tabla_dw["‖ΔW‖/‖W0‖"],
       color=tabla_dw["zona"].map(COLOR_ZONA))
for z in ("down", "mid", "up"):
    a1.bar([], [], color=COLOR_ZONA[z], label=z)          # solo la leyenda
a1.set(xlabel="capa adaptada (down → mid → up)",
       ylabel=r"$\|\Delta W\|_F\,/\,\|W_0\|_F$",
       title="Cuánto se movió cada capa")
a1.legend()
resumen = tabla_dw.pivot_table(index="zona", columns="tipo",
                               values="‖ΔW‖/‖W0‖", aggfunc="mean")
resumen.reindex([z for z in orden if z in resumen.index]).plot.bar(
    ax=a2, rot=0, title="promedio por zona y tipo")
plt.tight_layout()
plt.show()
display(tabla_dw.nlargest(5, "‖ΔW‖/‖W0‖")[["zona", "tipo", "‖ΔW‖/‖W0‖"]]
        .style.format({"‖ΔW‖/‖W0‖": "{:.2e}"}))

Al generar, la perilla aplica $W=W_0+\lambda\,\Delta W$.

| $\lambda$ | Qué aplica |
|---|---|
| $0$ | Ninguna corrección. |
| $0.5$ | La mitad de la corrección entrenada. |
| $1$ | La corrección entrenada. |
| $1.5$ | Una vez y media la corrección entrenada. |

**Cómo leer la malla.** Cada fila cambia $\lambda$. Dentro de una columna, el primer prompt y el ruido inicial se mantienen fijos. Una corrección más intensa puede hacer visible un cambio sutil o producir artefactos. La imagen se obtiene en varios pasos de denoising, así que el cambio visual puede ser no lineal.

La celda también compara, píxel por píxel, $\lambda=0$ con el adaptador apagado. Como la corrección se anula, se espera una diferencia nula o cercana a cero.

In [ ]:
LAMBDAS = [0.0, 0.5, 1.0, 1.5]
pares = combinar(PROMPTS[:1], P["semillas"][:2])   # el mismo prompt y semillas
imgs = []
for lam in LAMBDAS:
    imgs += generar(pares, lora=True, escala=lam)
malla(imgs, filas=len(LAMBDAS), titulo=f"La perilla λ · «{PROMPTS[0]}»",
      etiquetas=[f"λ = {lam}" for lam in LAMBDAS])

# λ = 0 contra el modelo base (adaptador apagado), píxel por píxel.
base = generar(pares, lora=False)
dif = [np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32))
       for a, b in zip(imgs[:len(pares)], base)]
print(f"λ = 0 contra el base: diferencia media {np.mean([d.mean() for d in dif]):.3f}"
      f", máxima {max(d.max() for d in dif):.0f} (escala 0–255)")

## 07 · Métricas, explicadas

Compara **base y LoRA en validation**. Si piensas probar ajustes, ejecuta hasta esta sección y reserva la 08 para tu configuración final. Para evaluar las generaciones, `generadas` recorre los captions del split y vuelve al primero si necesita más. Base y LoRA reciben las mismas parejas de caption y semilla.

CLIP representa imágenes y texto mediante embeddings. Con vectores normalizados $u$ y $v$, su similitud es $\cos(u,v)=u^\top v\in[-1,1]$. Un valor alto indica cercanía en el espacio de CLIP. Con pocas muestras, una sola imagen puede mover el promedio. Lee cada cifra junto con las mallas.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP = CLIPModel.from_pretrained(MODELO_CLIP).to(DEVICE).eval()
PROC = CLIPProcessor.from_pretrained(MODELO_CLIP)


def _vector(salida):
    """get_*_features devuelve un tensor o (versiones nuevas) un objeto."""
    if torch.is_tensor(salida):
        return salida
    for campo in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(salida, campo, None) is not None:
            return getattr(salida, campo)
    raise TypeError(type(salida))


@torch.no_grad()
def emb_imagenes(imgs):
    """(N, D) con norma 1: el coseno entre dos imágenes es su producto punto."""
    out = []
    for i in range(0, len(imgs), 32):
        x = PROC(images=imgs[i:i + 32], return_tensors="pt").to(DEVICE)
        out.append(_vector(CLIP.get_image_features(**x)).float().cpu())
    return F.normalize(torch.cat(out), dim=-1)


@torch.no_grad()
def emb_textos(textos):
    x = PROC(text=list(textos), return_tensors="pt", padding=True,
             truncation=True, max_length=77).to(DEVICE)
    return F.normalize(_vector(CLIP.get_text_features(**x)).float().cpu(), dim=-1)


def clip_score(imgs, textos, emb=None):
    """100 · coseno(imagen_i, texto_i), promediado sobre las parejas."""
    emb = emb_imagenes(imgs) if emb is None else emb
    return float(100 * (emb * emb_textos(textos)).sum(-1).mean())


def generadas(conjunto, semilla0):
    """P["n_gen"] imágenes con los captions de `conjunto` (en ciclo), base y
    LoRA con las MISMAS parejas (caption, semilla), y sus embeddings CLIP."""
    textos = TEXTOS[conjunto]
    pares = [(textos[i % len(textos)], semilla0 + i) for i in range(P["n_gen"])]
    gen = {"textos": [t for t, _ in pares], "emb": {}}
    for k, lora in (("base", False), ("LoRA", True)):
        gen[k] = generar(pares, lora=lora)
        gen["emb"][k] = emb_imagenes(gen[k])
    return gen


EMB_TRAIN = emb_imagenes(IMGS["train"])        # para estilo y memorización
GEN_VAL = generadas("validation", 1000)
print(f"{P['n_gen']} imágenes de validation generadas con el base y con LoRA")

**Loss de difusión**

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| MSE entre el ruido añadido a los latents reales y el ruido que predice el UNet. | $[0,\infty)$ | Una loss menor que la del base indica mejor denoising en esta evaluación. | Depende de los pares de ruido elegidos. Puede bajar aunque las generaciones sigan lejos del estilo buscado. |

Se reutiliza `perdida` de la sección 05 con los mismos pares $(t,\varepsilon)$ para ambos modelos.

In [ ]:
def m_loss(conjunto):
    """Loss de difusión con (t, ε) fijos, sin y con el adaptador."""
    return {"base": perdida(conjunto, lora=False),
            "LoRA": perdida(conjunto, lora=True)}


VAL = {"loss de difusión": m_loss("validation")}
print(pd.Series(VAL["loss de difusión"]).round(4).to_string())

**CLIP score**

Para cada pareja, compara el embedding de la imagen generada $g_i$ con el de su caption $c_i$. El código promedia los cosenos y multiplica por cien.

$$
\text{CLIP score}=\frac{100}{N}\sum_{i=1}^N\cos(g_i,c_i).
$$

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Correspondencia entre la imagen generada y el caption, según CLIP. | $[-100,100]$ | LoRA mantiene o mejora la correspondencia con respecto al base. | Con captions genéricos, una imagen genérica puede puntuar bien. CLIP puede pasar por alto detalles del prompt o del estilo. |

In [ ]:
def m_clip(gen):
    return {k: clip_score(None, gen["textos"], gen["emb"][k])
            for k in ("base", "LoRA")}


VAL["CLIP score"] = m_clip(GEN_VAL)
print(pd.Series(VAL["CLIP score"]).round(2).to_string())

**Parecido al estilo**

La celda promedia y normaliza los embeddings de las imágenes reales del split evaluado para obtener su centroide $m$. Luego promedia $\cos(g_i,m)$ sobre las imágenes generadas. La **referencia** calcula ese mismo promedio para las imágenes reales de **train** frente al mismo centroide.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Cercanía de las generaciones al centro del split real en el espacio de CLIP. | $[-1,1]$ | LoRA supera al base y se acerca a la referencia de train. | El centroide mezcla tema, encuadre y estilo. Repetir un motivo frecuente o copiar ejemplos puede elevar la cifra. |

In [ ]:
def m_estilo(conjunto, gen):
    """Coseno medio entre las generadas y el centroide CLIP de las reales
    de `conjunto`. Referencia: las reales de train contra ese centroide."""
    centro = F.normalize(emb_imagenes(IMGS[conjunto]).mean(0), dim=-1)
    salida = {k: float((gen["emb"][k] @ centro).mean()) for k in ("base", "LoRA")}
    salida["referencia"] = float((EMB_TRAIN @ centro).mean())
    return salida


VAL["parecido al estilo"] = m_estilo("validation", GEN_VAL)
print(pd.Series(VAL["parecido al estilo"]).round(3).to_string())

**Vecino más cercano en train**

Para cada imagen, toma el mayor coseno CLIP con una imagen real de **train** y promedia esos máximos. La **referencia** aplica la misma operación a imágenes reales del split evaluado, aquí **validation**.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Qué tan cerca suele quedar una generación de algún ejemplo de train. | $[-1,1]$ | Si LoRA se parece más al estilo y queda cerca de la referencia, hay menos indicios de copia. | En un dataset homogéneo, una imagen nueva también puede tener un vecino cercano. Un coseno alto indica similitud en CLIP. |

Si LoRA supera ampliamente la referencia, busca parecidos concretos con train antes de atribuirlo a memorización.

In [ ]:
def vecino(emb):
    """Coseno CLIP del vecino más cercano en train, para cada fila."""
    return (emb @ EMB_TRAIN.T).max(dim=1).values


def m_memoria(conjunto, gen):
    """Media del coseno al vecino de train: generadas (base, LoRA) y, como
    referencia, imágenes reales de `conjunto` que el modelo nunca vio."""
    salida = {k: float(vecino(gen["emb"][k]).mean()) for k in ("base", "LoRA")}
    salida["referencia"] = float(vecino(emb_imagenes(IMGS[conjunto])).mean())
    return salida


VAL["vecino en train"] = m_memoria("validation", GEN_VAL)
print(pd.Series(VAL["vecino en train"]).round(3).to_string())

**Olvido con el adaptador activo**

La celda compara el CLIP score de base y LoRA en tres prompts ajenos, con las mismas semillas para ambos modelos. Si alguno de los temas coincide con tu dataset, cambia `PROMPTS_AJENOS` antes de evaluar.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| Si las imágenes de temas ajenos siguen correspondiendo a sus prompts al activar LoRA. | Score en $[-100,100]$. La diferencia LoRA menos base está en $[-200,200]$. | Scores parecidos por prompt. | Cambiar el estilo también puede alterar el score. Una media estable puede ocultar un prompt que empeoró. |

In [ ]:
# Prompts que no tienen nada que ver con el dataset demo. Si tu dataset ES
# de perros, ciudades o fruta, cambia esta lista por otros temas ajenos.
PROMPTS_AJENOS = ["a photo of a dog", "a city street at night",
                  "a bowl of fruit"]


def m_olvido(semillas):
    """CLIP score de cada prompt ajeno, base vs LoRA, mismas semillas.
    Devuelve la tabla por prompt y las imágenes (prompt por prompt)."""
    pares = combinar(PROMPTS_AJENOS, semillas)
    imgs = {"base": generar(pares, lora=False), "LoRA": generar(pares, lora=True)}
    n, filas = len(semillas), []
    for j, prompt in enumerate(PROMPTS_AJENOS):
        filas.append({"prompt": prompt, **{
            k: clip_score(v[j * n:(j + 1) * n], [prompt] * n)
            for k, v in imgs.items()}})
    return pd.DataFrame(filas).set_index("prompt"), imgs


OLVIDO_VAL, _ = m_olvido(P["semillas"])
VAL["olvido"] = OLVIDO_VAL.mean().to_dict()
display(OLVIDO_VAL.assign(**{"LoRA − base": OLVIDO_VAL["LoRA"] -
                             OLVIDO_VAL["base"]}).round(2))

**Cómo leer la tabla.** La diferencia «LoRA − base» tiene un sentido distinto en cada fila. Una loss puede bajar, el parecido al estilo subir y el vecino en train mantenerse cerca de su referencia. Revisa también si los captions y los prompts ajenos se siguen respetando. Las dos referencias de la tabla responden a preguntas distintas, descritas junto a cada métrica.

**Cómo leer la malla.** Cada columna usa un caption de validation. Arriba ves la imagen real. En las dos filas inferiores, base y LoRA generan con el mismo caption y la misma semilla.

In [ ]:
BUENO = {"loss de difusión": "más baja",
         "CLIP score": "alto; que LoRA no caiga mucho",
         "parecido al estilo": "LoRA más alto que base",
         "vecino en train": "cerca de la referencia",
         "olvido": "LoRA ≈ base"}


def resumen(medidas):
    """Una fila por métrica: base, LoRA, referencia y qué es bueno."""
    tabla = pd.DataFrame(medidas).T.reindex(columns=["base", "LoRA",
                                                     "referencia"])
    tabla["LoRA − base"] = tabla["LoRA"] - tabla["base"]
    tabla["qué es bueno"] = pd.Series(BUENO)
    return tabla


display(resumen(VAL).style.format(
    {c: "{:.4f}" for c in ("base", "LoRA", "referencia", "LoRA − base")},
    na_rep="—").set_caption("VALIDATION · base vs LoRA"))
k = min(4, len(IMGS["validation"]), P["n_gen"])
malla(IMGS["validation"][:k] + GEN_VAL["base"][:k] + GEN_VAL["LoRA"][:k],
      filas=3, titulo="validation · real / base / LoRA (mismo caption y semilla)",
      etiquetas=["real", "base", "LoRA"])

## 08 · Testeo, una sola vez

> **Ejecuta la evaluación de test una sola vez**, con el adaptador y la configuración ya elegidos en validation. Si usas el resultado de test para ajustar el modelo, necesitarías otro conjunto reservado para medir el resultado final.

La primera tabla repite las cinco métricas con el split de **test** para la loss, el CLIP score, el parecido al estilo y el vecino en train. El olvido se mide con los mismos prompts ajenos y nuevas semillas. Las generaciones con captions de test también usan semillas nuevas. La segunda tabla compara las cifras de LoRA en validation y test.

In [ ]:
# Semillas 3000, 3001, … y 4000, …: ninguna sección anterior las usó.
GEN_TEST = generadas("test", 3000)
OLVIDO_TEST, OLVIDO_IMGS = m_olvido([4000 + i for i in range(len(P["semillas"]))])
TEST = {"loss de difusión": m_loss("test"),
        "CLIP score": m_clip(GEN_TEST),
        "parecido al estilo": m_estilo("test", GEN_TEST),
        "vecino en train": m_memoria("test", GEN_TEST),
        "olvido": OLVIDO_TEST.mean().to_dict()}
TABLA_TEST = resumen(TEST)
display(TABLA_TEST.style.format(
    {c: "{:.4f}" for c in ("base", "LoRA", "referencia", "LoRA − base")},
    na_rep="—").set_caption("TEST · base vs LoRA (una sola vez)"))
# Validation contra test, solo para LoRA: ¿fue validation un buen juez?
display(pd.DataFrame({"validation": resumen(VAL)["LoRA"],
                      "test": TABLA_TEST["LoRA"]}).round(4))

**Cómo leer las tablas.** En test, compara base y LoRA en cada fila y mira las referencias. Para estilo, son las imágenes reales de train frente al centroide de test. Para el vecino, son las imágenes reales de test frente a sus vecinos en train. Una brecha entre validation y test puede deberse a diferencias entre sus ejemplos o a la variabilidad de una muestra pequeña. Reporta lo observado junto con las mallas.

| Malla | Cómo leerla |
|---|---|
| **Test** | Cada columna muestra una imagen real y dos generaciones con su caption. Base y LoRA comparten el ruido inicial. |
| **Tus prompts** | Cada prompt tiene una fila «antes», recuperada de la sección 03, y una «después». Compara la misma columna para mantener fija la semilla. |

In [ ]:
# Test: real / base / LoRA con el mismo caption y semilla.
k = min(8 if LARGO else 4, len(IMGS["test"]), P["n_gen"])
malla(IMGS["test"][:k] + GEN_TEST["base"][:k] + GEN_TEST["LoRA"][:k],
      filas=3, titulo="test · real / base / LoRA (mismo caption y semilla)",
      etiquetas=["real", "base", "LoRA"])

# Tus prompts.txt: las MISMAS parejas de la sección 03, ahora con LoRA.
DESPUES = generar(combinar(PROMPTS, P["semillas"]), lora=True)
n = len(P["semillas"])
imgs, filas = [], []
for j, prompt in enumerate(PROMPTS):
    imgs += ANTES[j * n:(j + 1) * n] + DESPUES[j * n:(j + 1) * n]
    filas += [f"antes · {prompt}", f"después · {prompt}"]
malla(imgs, filas=2 * len(PROMPTS), titulo="Tus prompts: antes y después",
      etiquetas=filas)

**Cómo leer la malla.** Por cada prompt ajeno hay una fila del base y otra de LoRA. Compara dentro de la misma columna. Distingue entre un cambio de estilo y un caso en que desaparece el objeto o la escena que pediste.

La tabla muestra el CLIP score por prompt. Una diferencia «LoRA − base» negativa indica menor correspondencia según CLIP. Revisa la imagen para interpretar la caída. Esta prueba mide el comportamiento con el adaptador activo. Al apagarlo, vuelves a usar los pesos base.

In [ ]:
n = len(P["semillas"])
imgs, filas = [], []
for j, prompt in enumerate(PROMPTS_AJENOS):
    for quien in ("base", "LoRA"):
        imgs += OLVIDO_IMGS[quien][j * n:(j + 1) * n]
        filas.append(f"{quien} · {prompt}")
malla(imgs, filas=len(filas), titulo="Prompts ajenos: base y LoRA",
      etiquetas=filas)
display(OLVIDO_TEST.assign(**{"LoRA − base": OLVIDO_TEST["LoRA"] -
                              OLVIDO_TEST["base"]}).round(2))

La celda guarda los pesos del adaptador en `mi_lora/pytorch_lora_weights.safetensors`, en formato diffusers. Imprime el tamaño real del archivo junto a la estimación de la sección 04 y los MB del UNet en fp16.

En Colab se inicia la descarga. Si el navegador la bloquea, abre la carpeta `mi_lora` en el panel **📁 Archivos** y selecciona «Descargar» sobre el archivo. Fuera de Colab, la celda imprime la ruta donde quedó. Conserva el adaptador junto con el nombre del modelo base que usaste para entrenarlo.

In [ ]:
from diffusers.utils import convert_state_dict_to_diffusers
from peft.utils import get_peft_model_state_dict

CARPETA_LORA = Path("mi_lora")
pesos = convert_state_dict_to_diffusers(get_peft_model_state_dict(UNET))
StableDiffusionPipeline.save_lora_weights(CARPETA_LORA, unet_lora_layers=pesos,
                                          safe_serialization=True)
ARCHIVO_LORA = CARPETA_LORA / "pytorch_lora_weights.safetensors"
mb = ARCHIVO_LORA.stat().st_size / 1e6
print(f"{ARCHIVO_LORA}: {mb:.1f} MB (la cuenta de la sección 04 decía "
      f"{4 * N_LORA / 1e6:.1f} MB) · el UNet en fp16: {2 * N_UNET / 1e6:,.0f} MB")
try:                                  # en Colab: lo baja tu navegador
    from google.colab import files
    files.download(str(ARCHIVO_LORA))
except ImportError:
    print("(fuera de Colab: el archivo quedó en", ARCHIVO_LORA.resolve(), ")")

Usa esta receta **durante la revisión en validation**, antes de abrir test. Cambia una cosa a la vez y vuelve a ejecutar desde la sección que la configura.

| Lo que ves | Qué probar |
|---|---|
| El estilo apenas cambia y se siguen respetando los captions. | Prueba más pasos. Usa la perilla para ver si al amplificar la corrección aparece el cambio que buscabas. |
| Aparecen artefactos y sube la loss de validation. | Prueba un learning rate menor. |
| El vecino en train queda muy por encima de la referencia y ves ejemplos casi repetidos. | Prueba menos pasos o añade imágenes variadas. |
| Las salidas pierden detalles que pediste en el prompt. | Revisa si los captions de train describen esos detalles. Prueba captions más específicos. |
| Los prompts ajenos pierden el objeto o la escena. | Prueba una escala $\lambda$ menor al generar o menos pasos de entrenamiento. |
| Baja la loss, pero el estilo sigue lejos. | Revisa los recortes, la coherencia del dataset y los captions. Juzga el estilo con las mallas y el parecido al estilo. |
| Se agota la memoria durante el entrenamiento. | Reduce `P["batch"]`. Reinicia el entorno para liberar memoria y vuelve a ejecutar. |